# contextvars

对应 `stdlib.md`：当前这段执行里的变量；任务复制一份，线程默认不继承。

笔记本在 `python_stdlibs/contextvars/qa.ipynb`。需要读写文件时，工作空间就是这个目录。

先运行下一格，得到 `ROOT`。每题只改 `# 作答` 下面的代码。前置代码不用改。做完自己跑通即可，先不要对答案。

`contextvars` 存的是当前这段执行里的值。它不是 `contextlib`（那个管 `with`）。


In [1]:
from pathlib import Path

def lab_root() -> Path:
    """qa.ipynb 所在目录，即 python_stdlibs/contextvars。"""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if folder.name == "contextvars" and (folder / "qa.ipynb").is_file():
            return folder
        candidate = folder / "codes" / "python_stdlibs" / "contextvars"
        if (candidate / "qa.ipynb").is_file():
            return candidate
    return here

ROOT = lab_root()
ROOT


PosixPath('/Users/keyficller/Documents/AEFS-Notes/codes/python_stdlibs/contextvars')


## 1. 默认值和写入

`ContextVar` 带上 `default="anon"`。先 `get`，再 `set("ada")`，再 `get`。两行都打出来。


In [2]:
# 作答

import contextvars

user = contextvars.ContextVar("user", default="anon")
print(user.get())
user.set("ada")
print(user.get())
#评阅
# 对。没 set 过时 get 返回构造时的 default。set 只改当前这段执行。

#参考答案
# user = contextvars.ContextVar("user", default="anon")
# print(user.get())
# user.set("ada")
# print(user.get())


anon
ada


## 2. 退回上一层

先 `set("ada")`，再 `set("grace")`。`reset` 最近一次的 `Token`，把当前值打出来。


In [3]:
# 作答

import contextvars

user = contextvars.ContextVar("user")
first = user.set("ada")
second = user.set("grace")
user.reset(second)
print(user.get())
#评阅
# 对。reset 最近的 Token，回到上一层的值。reset 更早的 Token 会把中间的写入一起退掉。

#参考答案
# user = contextvars.ContextVar("user")
# first = user.set("ada")
# second = user.set("grace")
# user.reset(second)
# print(user.get())


ada


## 3. 快照看到复制时的值

`set("red")` 之后 `copy_context`。外面再 `set("green")`。用快照的 `run` 打印里面的值，再打印外面的值。


In [4]:
# 作答

import contextvars

color = contextvars.ContextVar("color", default="blue")
color.set("red")
snap = contextvars.copy_context()
color.set("green")

def show():
    print(color.get())

snap.run(show)
print(color.get())
#评阅
# 对。copy_context 复制的是当时的值。外面之后的 set 改不到这份快照。

#参考答案
# color.set("red")
# snap = contextvars.copy_context()
# color.set("green")
# snap.run(show)  # red
# print(color.get())  # green


red
green


## 4. 快照里的写入不回到外面

当前值是 `1`。复制快照后，在 `run` 里加上 `10` 并打印。再打印外面的值，以及快照里的值。


In [5]:
# 作答

import contextvars

n = contextvars.ContextVar("n", default=0)
n.set(1)
snap = contextvars.copy_context()

def bump():
    n.set(n.get() + 10)
    print("inside", n.get())

snap.run(bump)
print("outer", n.get())
print("snap", snap.get(n))
#评阅
# 对。run 里面的 set 留在快照上。调用方仍是 1。Context.get 读的是快照，不是当前上下文。

#参考答案
# snap.run(bump)
# print("outer", n.get())
# print("snap", snap.get(n))


inside 11
outer 1
snap 11


## 5. 任务复制创建那一刻

`set("parent")` 之后 `asyncio.create_task`。父任务再 `set("later")`。打印任务读到的值，再打印父任务当前的值。


In [6]:
# 作答

import asyncio
import contextvars

job = contextvars.ContextVar("job", default="none")

async def main():
    job.set("parent")

    async def task():
        return job.get()

    handle = asyncio.create_task(task())
    job.set("later")
    print(await handle)
    print(job.get())

asyncio.run(main())
#评阅
# 对。create_task 在创建那一刻复制上下文。父任务之后的 set，子任务看不到。

#参考答案
# handle = asyncio.create_task(task())
# job.set("later")
# print(await handle)  # parent
# print(job.get())  # later


parent
later


## 6. 线程默认不继承

当前值是 `"parent"`，且这个变量没有 `default`。新线程里 `get`，抓到 `LookupError` 就记下 `"missing"`。再用 `copy_context().run` 起一个线程，记下它读到的值。两行都打出来。


In [7]:
# 作答

import contextvars
import threading

job = contextvars.ContextVar("job")
job.set("parent")
box = {}

def plain():
    try:
        box["plain"] = job.get()
    except LookupError:
        box["plain"] = "missing"

def copied():
    box["copied"] = job.get()

bare = threading.Thread(target=plain)
bare.start()
bare.join()

ctx = contextvars.copy_context()
carried = threading.Thread(target=ctx.run, args=(copied,))
carried.start()
carried.join()
print(box["plain"])
print(box["copied"])
#评阅
# 对。新线程是空上下文，看不到父线程 set 的值。copy_context().run 才把快照带进去。

#参考答案
# threading.Thread(target=plain).start()
# ctx = contextvars.copy_context()
# threading.Thread(target=ctx.run, args=(copied,))


missing
parent
